# Phase 1 학습 — Colab / Kaggle GPU

맥북(MPS)에서는 epoch당 약 19분이라 50 epoch에 16시간이 걸린다.
T4 GPU + 혼합정밀이면 훨씬 빠르다.

**수집은 로컬에서만 한다** (키움 API는 등록된 IP에서만 호출된다).
여기서는 이미 만들어진 `train_bundle.zip`만 올려서 학습만 돌린다.

---
### 사용 순서
1. 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택
2. 아래 셀을 위에서부터 실행
3. 2번 셀에서 로컬의 `outputs/train_bundle.zip`을 업로드
4. 학습이 끝나면 마지막 셀로 체크포인트를 내려받아 로컬 `outputs/checkpoints/`에 둔다


## 1. GPU 확인


In [ ]:
!nvidia-smi -L || echo 'GPU 없음 — 런타임 유형을 GPU로 바꿀 것'
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. 코드 받기 + 데이터 업로드

코드는 GitHub에서 클론한다. 데이터(`data/`)는 저장소에 없으므로 직접 올린다.


In [ ]:
!git clone -q https://github.com/sungjunpk/Capstone_Stock_Price_Prediction.git
%cd Capstone_Stock_Price_Prediction
!pip -q install pyarrow pyyaml python-dotenv


In [ ]:
# 로컬에서 `python scripts/package_data.py` 로 만든 train_bundle.zip 을 올린다 (약 35MB)
from google.colab import files
up = files.upload()

import zipfile
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('.')
!ls -lh data/processed/


## 3. 학습

`--batch-size`를 GPU 메모리에 맞춰 올린다. T4(16GB)면 256~512가 무난하다.
코드가 CUDA를 감지하면 혼합정밀(AMP)과 DataLoader 워커를 자동으로 켠다.


In [ ]:
!python scripts/train.py --batch-size 256 --lr 6e-4


### 배관만 먼저 확인하고 싶다면
전체를 돌리기 전에 소규모로 한 번 돌려보는 편이 안전하다.


In [ ]:
!python scripts/train.py --smoke


## 4. 결과 내려받기

체크포인트와 실험 리포트를 로컬로 가져와 같은 경로에 둔다.
리포트에는 VSN 피처 중요도가 들어 있어 해석가능성 분석에 쓴다.


In [ ]:
from google.colab import files
import glob, os

for p in ['outputs/checkpoints/phase1_best.pt', *sorted(glob.glob('outputs/reports/*.json'))[-1:]]:
    if os.path.exists(p):
        print('다운로드:', p, f'({os.path.getsize(p)/1e6:.1f}MB)')
        files.download(p)


---
## Kaggle 로 돌릴 때 다른 점

- 데이터는 **Datasets → New Dataset** 으로 `train_bundle.zip` 을 올린 뒤
  노트북 우측 *Add Input* 으로 붙인다. 경로는 `/kaggle/input/<데이터셋이름>/`
- 업로드 셀 대신 아래처럼 복사한다

```python
!cp -r /kaggle/input/<데이터셋이름>/data .
```

- Kaggle 은 주당 30시간 GPU 무료이고 세션이 12시간까지 유지되어
  Colab 무료 티어보다 긴 학습에 유리하다.
- 인터넷 사용은 노트북 설정에서 **Internet: On** 을 켜야 `git clone` 이 된다.
